# Clase 14 · Notebook 02
# Pooling y las capas Conv2D / MaxPooling2D en Keras

**Introducción al Deep Learning — Módulo 3, Unidad 1: Redes convolucionales (Parte I)**

Cerramos los fundamentos:

1. El **pooling** (max y average) desde cero.
2. Las capas reales de Keras: **Conv2D** y **MaxPooling2D**, y cómo cambian el tamaño según el **padding**.

(Aún no entrenamos una red: eso será en la Parte II. Aquí entendemos las piezas.)

> 💡 En Colab, TensorFlow ya viene instalado. Ejecuta las celdas en orden con **Shift + Enter**.

## 1. El pooling desde cero (max vs average)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from scipy.ndimage import zoom
np.random.seed(42)

img = zoom(load_digits().images[0], 6, order=1)   # imagen 48x48

def pooling(imagen, tam=2, modo="max"):
    oh, ow = imagen.shape[0] // tam, imagen.shape[1] // tam
    salida = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            region = imagen[i*tam:(i+1)*tam, j*tam:(j+1)*tam]
            salida[i, j] = region.max() if modo == "max" else region.mean()
    return salida

mx = pooling(img, 2, "max")
av = pooling(img, 2, "avg")
print("Original:", img.shape, "-> pooling 2x2:", mx.shape)

fig, ax = plt.subplots(1, 3, figsize=(11, 4))
ax[0].imshow(img, cmap="gray"); ax[0].set_title("Original")
ax[1].imshow(mx, cmap="gray");  ax[1].set_title("Max-pooling 2x2")
ax[2].imshow(av, cmap="gray");  ax[2].set_title("Average-pooling 2x2")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

El pooling **reduce el tamaño a la mitad** conservando la información principal. El max-pooling resalta
los valores fuertes; el average-pooling suaviza.

## 2. Las capas de Keras: Conv2D y MaxPooling2D

Keras espera las imágenes con forma `(alto, ancho, canales)`. Veamos cómo cada capa transforma ese tamaño.
Una `Conv2D` con 8 filtros 3×3 produce 8 mapas de características.

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
from tensorflow.keras.layers import Conv2D, MaxPooling2D

x = tf.random.normal((1, 28, 28, 1))   # una imagen 28x28 de 1 canal
print("Entrada:                         ", x.shape)

conv_valid = Conv2D(8, (3, 3), padding="valid")(x)
print("Conv2D 8x(3x3) padding='valid':  ", conv_valid.shape, " (se encoge a 26x26, 8 canales)")

conv_same = Conv2D(8, (3, 3), padding="same")(x)
print("Conv2D 8x(3x3) padding='same':   ", conv_same.shape, " (conserva 28x28, 8 canales)")

pooled = MaxPooling2D((2, 2))(conv_same)
print("MaxPooling2D 2x2:                ", pooled.shape, " (la mitad: 14x14)")

Observa:
- `padding='valid'` encoge la imagen; `padding='same'` la conserva.
- `Conv2D` con 8 filtros crea **8 canales** de salida (8 mapas de características).
- `MaxPooling2D 2x2` reduce alto y ancho a la mitad.

## 3. Encadenar capas: cómo evoluciona el tamaño

Así se reduce el tamaño y aumenta el número de mapas a medida que avanzamos en una CNN.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input

modelo = Sequential([
    Input(shape=(28, 28, 1)),
    Conv2D(8, (3, 3), padding="same", activation="relu"),
    MaxPooling2D((2, 2)),
    Conv2D(16, (3, 3), padding="same", activation="relu"),
    MaxPooling2D((2, 2)),
])
modelo.summary()
print("\nNota: el pooling no tiene parámetros que entrenar.")

## Resumen

- El **pooling** (max / average) reduce el tamaño conservando lo esencial; no tiene parámetros que entrenar.
- **Conv2D** aplica filtros (cada filtro = un canal de salida); `padding` decide si la imagen se encoge.
- **MaxPooling2D** reduce alto y ancho a la mitad.
- Encadenando conv + pooling, el tamaño baja y el número de mapas sube: así se extraen características.

Con esto tenemos todas las piezas. En **Redes convolucionales II** las uniremos en una CNN completa, con
aumentación de datos, para clasificar imágenes.